# 04 — Localizar os CEPs que sobraram (pelo endereço)

**Problema:** alguns CEPs da RAIS não ganham ponto pela base de logradouros (etapa 03) nem pelos pontos do CNEFE com o próprio CEP (classe P da etapa 05):
- **grandes usuários** (sufixo 900 a 999: empresas, órgãos, condomínios);
- CEPs dos Correios **sem trecho** na base e **sem pontos** no CNEFE (classes C e D);
- CEPs **fora da lista dos Correios** (CEPs antigos, alterados etc.) e sem pontos no CNEFE.

**Solução:** descobrir o **endereço** do CEP e localizar esse endereço, do mais confiável ao menos confiável. Antes de localizar, confere a **cidade** devolvida pelo ViaCEP/BrasilAPI: CEP de outro município não ganha ponto.

**1. Endereço do CEP** (`fonte_endereco`): Correios (lista da etapa 02) → **ViaCEP** → **BrasilAPI**. Grande usuário que já tem pontos no CNEFE com o próprio CEP **não consulta API** (usa os pontos). Do texto vêm a rua, o bairro e um número de referência:
- grande usuário: o número do prédio (`complemento` do ViaCEP);
- CEP de faixa (`de 1001/1002 ao fim`, `até 999/1000`): o meio da faixa (ou o início, se for "ao fim").

**2. Coordenada** (`fonte_coord`, na ordem):

| Fonte | Como | Confiança |
|---|---|---|
| `cnefe_cep_numero` | endereço do CNEFE com o mesmo CEP e o mesmo número (só número de prédio) | alta |
| `cnefe_endereco` | endereço do CNEFE na mesma rua (nome compatível) e com o mesmo número | alta (média sem bairro) |
| `cnefe_cep` | mediana dos endereços do CNEFE com o CEP | média |
| `cnefe_rua_numero_proximo` | mesma rua, número mais próximo | média (baixa sem bairro) |
| `cnefe_rua` | endereço do CNEFE mais perto do centro dos endereços da rua | média (baixa sem bairro) |
| `geosanja_nome` | trechos da base com nome compatível: ponto da linha mais perto do centro | média (baixa sem bairro) |
| `geocoding_numero` / `geocoding_rua` | **Nominatim** (OpenStreetMap), só se a rua devolvida tiver nome compatível | média / baixa |
| `cnefe_bairro` | **último recurso:** centro dos endereços do CNEFE cujo loteamento é compatível com o bairro do CEP (só se concentrados, ≤ `cnefe.dispersao_media_m`) | baixa |
| `cnefe_cep_vizinhos` | **último recurso:** centro dos endereços do CNEFE de CEPs com os mesmos 6 primeiros dígitos (≥ 3 locais, concentrados). Não vale para CEP terminado em `000` | baixa |
| `cnefe_ceps_desmembrados` | CEP **desmembrado** em vários CEPs novos (informado em `cep_novo`): centro dos endereços do CNEFE dos CEPs novos | média (até `cnefe.dispersao_media_m` de dispersão) / baixa |
| `nao_localizavel` | **você** marca no CSV: CEP genérico do município, desativado etc. Não ganha ponto e sai da lista manual | — |
| `outra_cidade` | o ViaCEP/BrasilAPI diz que o CEP é de **outro município** (o prefixo `12` cobre todo o Vale): não é localizado e sai da lista manual | — |
| `sem_coordenada` | nada achou: preencher `lat`/`lon` e `fonte_coord = manual` | — |

Os dois últimos recursos só são usados quando todas as consultas foram concluídas (se uma API falhou, o CEP fica sem coordenada para ser refeito depois, em vez de ganhar um ponto aproximado).

**Rua com o mesmo nome em vários lugares:** fica só a parte cujo **loteamento** (Geosanja) é compatível com o **bairro** do endereço. Sem bairro compatível, a rua só é aceita se os pontos estiverem concentrados (dispersão ≤ `cnefe.dispersao_media_m`); senão, passa para a próxima fonte.

Todas as coordenadas precisam cair dentro da área do município.

**O que você pode escrever no CSV** (linha do CEP em `ceps_localizados.csv`; vale nas próximas execuções; para refazer um CEP que já tem coordenada, apague `lat` e `lon`):
- **Endereço que as APIs não acharam:** preencha `logradouro`, `bairro` (e `cidade`, se quiser) e `fonte_endereco = manual`. A cascata roda com esse endereço, sem consultar as APIs;
- **CEP trocado por outro:** `cep_novo` com 1 CEP (as consultas passam a usar o novo);
- **CEP desmembrado em vários:** `cep_novo` com 2 ou mais CEPs separados por `|` (ex.: `12225011|12225732`);
- **CEP sem como localizar** (genérico do município, desativado): `fonte_coord = nao_localizavel` e, em `obs`, o motivo. Se for de outra cidade, `fonte_coord = outra_cidade` e `cidade`;
- **Coordenada que você mesmo achou:** `lat`, `lon` e `fonte_coord = manual`.

**Cache das APIs (parquet):** toda resposta definitiva (achou ou "não encontrado") de ViaCEP, BrasilAPI e Nominatim é gravada **na hora** em `03_processamento\04_localizar_ceps\cache_viacep.parquet`, `cache_brasilapi.parquet` e `cache_nominatim.parquet`. Se a etapa for interrompida (Ctrl+C, queda de rede, bloqueio), nada se perde; na próxima execução só se consulta o que falta. Falhas de rede **não** entram no cache (são tentadas de novo). Para refazer as consultas, apague o parquet do serviço.

**Bloqueio das APIs:** depois de `localizacao.falhas_seguidas_max` falhas de rede seguidas, aquele serviço é desligado até o fim da execução (sem insistir). A etapa continua com o que tem, **avisa no log quantas consultas não foram feitas** e marca no `obs` do CSV os CEPs afetados. Basta rodar a etapa de novo mais tarde.

**Coordenada da BrasilAPI não é usada** (pouco confiável): a BrasilAPI serve só para o endereço.

**Saída (editável):** `02_dados_editados\ceps_localizados.csv` (separador `;`). Funciona como **cache**: linhas com coordenada ou `manual` não são refeitas. Para refazer tudo, apague o CSV. Antes de regravar, cópia em `02_dados_editados\_backup\`. Na primeira execução, as linhas `manual` do antigo `grande_usuario_ceps.csv` são trazidas.

**Usado por:** etapa 05 (classe G = localizado pelo endereço).

**Estrutura:** 1 Preparação · 2 Funções · 3 CEPs alvo e CSV · 4 Endereço do CEP · 5 CNEFE e Geosanja · 6 Nomes compatíveis · 7 Localizar · 8 Exportação · 9 Resumo

## 1. Preparação e configuração

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "04_localizar_ceps"                           # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import re                             # busca de padrões em texto
import json                           # guardar/ler respostas das APIs no cache
import time                           # pausas entre consultas
from functools import lru_cache       # guarda resultados de funções já calculadas
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import requests                       # consultas às APIs
import pyogrio                        # leitura de arquivos geográficos
import geopandas as gpd               # tabelas com geometria
import shapely                        # operações com geometrias
import shapely.ops                    # ponto mais próximo de uma linha
from pyproj import Transformer        # conversão de coordenadas (metros <-> graus)
from rapidfuzz import fuzz, process   # comparação de textos "parecidos"
from src import fontes                # arquivos editados (prioridade e backup)

# --- Entradas e saída (config.yaml) ----------------------------------------------------------------
ARQ_RAIS = cfg.caminho("processamento.rais_consolidada")     # rais consolidada (etapa 01)
ARQ_BASE = fontes.caminho_editado(cfg, "logradouros_cep")    # base de logradouros (etapa 03 ou editada)
CAMADA_BASE = cfg.camada("editados.logradouros_cep")
ARQ_LOCAIS = cfg.caminho("processamento.cnefe_locais")       # locais do cnefe e o segmento de cada um (etapa 03)
ARQ_CNEFE = cfg.caminho("brutos.cnefe")                      # cnefe do município (com número)
ARQ_PONTE = cfg.caminho("processamento.ponte")               # CEPs dos correios pareados (etapa 02)
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")             # CEPs dos correios sem segmento (etapa 02)
ARQ_CSV = cfg.caminho("editados.ceps_localizados")           # CSV editável (02_dados_editados)
ARQ_CSV_ANTIGO = cfg.pasta("editados") / "grande_usuario_ceps.csv"   # CSV da versão anterior (só para trazer o 'manual')

# --- Parâmetros ----------------------------------------------------------------------------------------
PREFIXO_MUNICIPIO = cfg.projeto("prefixo_cep")                             # só CEPs do município
CRS_METRICO = cfg.projeto("crs_metrico")
CRS_GRAUS = cfg.projeto("crs_graus")
SUFIXO_ESPECIAL_MIN = cfg.parametro("localizacao.sufixo_especial_min")    # sufixo >= isso = grande usuário
PAUSA_S = cfg.parametro("localizacao.pausa_s")                            # pausa entre consultas ao ViaCEP/BrasilAPI
FALHAS_MAX = cfg.parametro("localizacao.falhas_seguidas_max")             # falhas de rede seguidas que desligam um serviço
PAUSA_NOMINATIM_S = cfg.parametro("localizacao.pausa_nominatim_s")        # o Nominatim aceita no máximo 1 consulta por segundo
TIMEOUT_S = cfg.parametro("localizacao.timeout_s")                        # tempo máximo de espera por resposta
TENTATIVAS = cfg.parametro("localizacao.tentativas")                      # tentativas em caso de erro de rede
MARGEM_GRAUS = cfg.parametro("localizacao.margem_graus")                  # folga ao redor da área do cnefe
USER_AGENT = cfg.parametro("localizacao.user_agent")                      # identificação exigida pelo Nominatim
CEPS_SUBSTITUIDOS = {str(k): str(v) for k, v in (cfg.parametro("localizacao.ceps_substituidos") or {}).items()}   # antigo -> novo
LIMIAR_NOME = cfg.parametro("cnefe.limiar_nome")                          # nome de rua compatível
LIMIAR_BAIRRO = cfg.parametro("pareamento.limiar_bairro")                 # bairro x loteamento compatível
DISPERSAO_MAX_M = cfg.parametro("cnefe.dispersao_media_m")               # rua sem bairro compatível só vale se concentrada
URL_VIACEP = "https://viacep.com.br/ws/{cep}/json/"
URL_BRASILAPI = "https://brasilapi.com.br/api/cep/v2/{cep}"
URL_NOMINATIM = "https://nominatim.openstreetmap.org/search"
CIDADE = "São José dos Campos, SP, Brasil"                                # complemento das buscas no Nominatim
MUNICIPIO = "São José dos Campos"                                         # para conferir a cidade devolvida pelas APIs
COLUNAS_CNEFE = ["CEP", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "NUM_ENDERECO", "LATITUDE", "LONGITUDE"]
COLUNAS_CSV = ["cep", "cep_novo", "grupo", "unidade", "logradouro", "complemento", "numero_ref", "bairro", "cidade", "fonte_endereco",
               "lat", "lon", "fonte_coord", "confianca", "n_pontos", "obs", "n_estab_soma_anos", "vinc_ativos_soma_anos"]
GENERICOS_BAIRRO = {"VILA", "JARDIM", "PARQUE", "RESIDENCIAL", "CONDOMINIO", "CONJUNTO", "HABITACIONAL",
                    "LOTEAMENTO", "CHACARA", "CHACARAS", "SITIO", "BAIRRO"}

log.info("Configuração carregada!")
for _nome, _arq in [("rais consolidada", ARQ_RAIS), ("Base de logradouros", ARQ_BASE), ("Locais do cnefe", ARQ_LOCAIS),
                    ("cnefe", ARQ_CNEFE), ("Ponte (correios)", ARQ_PONTE), ("Órfãos (correios)", ARQ_ORFAOS),
                    ("CSV de CEPs localizados", ARQ_CSV), ("CSV antigo de grandes usuários", ARQ_CSV_ANTIGO)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())

## 2. Funções auxiliares
- `pedir_json` e `ServicoAPI`: chamada a uma API com novas tentativas, **cache em parquet** e parada automática se o serviço bloquear;
- `separar_faixa` e `numero_referencia`: tiram do texto do endereço a rua e o número de referência;
- `bairro_compativel`: bairro do endereço x loteamento do Geosanja;
- `escolher_lugar`: entre ruas de mesmo nome, fica com a do bairro certo (ou recusa, se for ambíguo).

In [ ]:
from src.validacao import validar_tabela                         # bloco padrão de validação
from src.texto import normalizar, tirar_tipo                     # normalização de nomes de logradouro

para_graus = Transformer.from_crs(CRS_METRICO, CRS_GRAUS, always_xy=True)   # metros -> graus (x, y -> lon, lat)


def pedir_json(url, params, pausa):
    """Faz GET com tentativas. Devolve (dados, status): status = ok | nao_encontrado | erro_rede."""
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = requests.get(url, params=params, timeout=TIMEOUT_S, headers={"User-Agent": USER_AGENT})
            time.sleep(pausa)                                        # respeita o limite de cada serviço
            if resposta.status_code == 404:
                return None, "nao_encontrado"
            resposta.raise_for_status()                              # erro HTTP vira exceção
            return resposta.json(), "ok"
        except Exception as erro:                                     # rede caiu, tempo esgotado, bloqueio etc.
            log.warning("  falha (tentativa %d/%d) em %s: %s", tentativa, TENTATIVAS, url, erro)
            time.sleep(pausa * tentativa)
    return None, "erro_rede"


class ServicoAPI:
    """Uma API externa com cache em parquet e 'disjuntor'.

    - cache: respostas definitivas (ok / nao_encontrado) ficam no disco, gravadas a cada consulta nova;
    - disjuntor: depois de FALHAS_MAX falhas de rede seguidas, o serviço é desligado até o fim da execução;
    - 'pendentes': chaves que NÃO puderam ser consultadas (para avisar no final).
    """
    def __init__(self, nome, pausa_s):
        self.nome, self.pausa_s = nome, pausa_s
        self.arquivo = PASTA_SAIDA / f"cache_{nome}.parquet"
        self.cache, self.pendentes = {}, set()                      # cache: chave -> (status, dados)
        self.falhas_seguidas, self.desligado = 0, False
        self.n_rede = self.n_cache = 0
        if self.arquivo.exists():
            tab = pd.read_parquet(self.arquivo)
            self.cache = {c: (s, json.loads(j) if j else None) for c, s, j in zip(tab["chave"], tab["status"], tab["json"])}
        log.info("Cache %s: %d respostas guardadas (%s)", nome, len(self.cache), self.arquivo.name)

    def salvar(self):
        """Regrava o parquet do cache (por arquivo temporário, para não corromper se for interrompido)."""
        tab = pd.DataFrame({"chave": list(self.cache), "status": [v[0] for v in self.cache.values()],
                            "json": [json.dumps(v[1], ensure_ascii=False) if v[1] is not None else "" for v in self.cache.values()]})
        tmp = self.arquivo.with_suffix(".tmp")
        tab.to_parquet(tmp, index=False)
        tmp.replace(self.arquivo)

    def consultar(self, chave, url, params=None):
        """Devolve (dados, status): ok | nao_encontrado | erro_rede | indisponivel (serviço desligado)."""
        if chave in self.cache:                                      # já consultado antes: sem rede, sem pausa
            self.n_cache += 1
            status, dados = self.cache[chave]
            return dados, status
        if self.desligado:
            self.pendentes.add(chave)
            return None, "indisponivel"
        dados, status = pedir_json(url, params, self.pausa_s)
        self.n_rede += 1
        if status == "erro_rede":
            self.pendentes.add(chave)
            self.falhas_seguidas += 1
            if self.falhas_seguidas >= FALHAS_MAX:
                self.desligado = True
                log.warning("!!! %s DESLIGADO nesta execução (%d falhas seguidas). As consultas restantes NÃO serão feitas; "
                            "rode a etapa de novo mais tarde (o cache evita repetir o que já foi feito).", self.nome, FALHAS_MAX)
            return None, status
        self.falhas_seguidas = 0
        self.cache[chave] = (status, dados)
        self.salvar()                                                # gravado na hora: interromper não perde nada
        return dados, status


FALHAS = ("erro_rede", "indisponivel")                              # status que significam "não consegui consultar"
VIACEP = ServicoAPI("viacep", PAUSA_S)
BRASILAPI = ServicoAPI("brasilapi", PAUSA_S)
NOMINATIM = ServicoAPI("nominatim", PAUSA_NOMINATIM_S)
SERVICOS = [VIACEP, BRASILAPI, NOMINATIM]


def nome_chave_txt(texto):
    """Nome de rua comparável de UM texto: normalizado e sem o tipo (RUA, AVENIDA...)."""
    return tirar_tipo(normalizar(pd.Series([texto or ""])).iloc[0])


def separar_faixa(texto):
    """'Avenida X - de 1001/1002 ao fim' -> ('Avenida X', 'de 1001/1002 ao fim'). Sem ' - ' a faixa fica vazia."""
    partes = re.split(r"\s+-\s+", str(texto or "").strip(), maxsplit=1)
    return partes[0], (partes[1] if len(partes) > 1 else "")


def numero_referencia(texto, grande_usuario):
    """Número de referência do endereço. Devolve (número ou None, tipo): tipo = predio | faixa | ''.

    Grande usuário: o primeiro número do complemento (número do prédio).
    Faixa ('de 1001/1002 a 1999/2000', 'até 999/1000', 'de 6001/6002 ao fim'): o meio da faixa,
    o início se for 'ao fim', a metade se for 'até'. Pares ímpar/par (1001/1002) contam como um número.
    Texto com 'km' (rodovias) não gera número.
    """
    t = str(texto or "").lower()
    if not t or "km" in t:
        return None, ""
    nums = [int(x) for x in re.findall(r"\d+", t)]
    if not nums:
        return None, ""
    if grande_usuario:
        return nums[0], "predio"
    limpos = [n for i, n in enumerate(nums) if i == 0 or n != nums[i - 1] + 1]   # tira o par do lado oposto (1001/1002)
    if "fim" in t:
        return limpos[0], "faixa"
    if len(limpos) >= 2:
        return (limpos[0] + limpos[-1]) // 2, "faixa"
    if "até" in t or "ate " in t:
        return max(limpos[0] // 2, 1), "faixa"
    return limpos[0], "faixa"


@lru_cache(maxsize=None)
def ler_ceps_novos(texto):
    """Da célula 'cep_novo': lista de CEPs de 8 dígitos. Aceita '12225-011|12225732' (hífen e separadores livres)."""
    return re.findall(r"\d{8}", re.sub(r"(?<=\d)[-.](?=\d)", "", str(texto)))


@lru_cache(maxsize=None)
def cidade_chave(texto):
    """Cidade normalizada (sem acento, caixa alta, sem preposições), para comparar 'São José dos Campos' com a resposta da API."""
    return normalizar(pd.Series([texto])).iloc[0]


def de_outro_municipio(cidade):
    """True se a API informou uma cidade e ela NÃO é o município do projeto."""
    return bool(cidade) and cidade_chave(cidade) != cidade_chave(MUNICIPIO)


def bairro_compativel(bairro_key, lote_key):
    """True se o bairro (normalizado) e o loteamento (normalizado) são o mesmo lugar."""
    if not bairro_key or not lote_key:
        return False
    a = " ".join(p for p in bairro_key.split() if p not in GENERICOS_BAIRRO)   # 'VILA BETANIA' -> 'BETANIA'
    b = " ".join(p for p in lote_key.split() if p not in GENERICOS_BAIRRO)
    if a and a == b:
        return True
    return fuzz.ratio(bairro_key, lote_key) >= LIMIAR_BAIRRO


def dispersao(df):
    """Mediana da distância (m) de cada ponto ao centro (mediana) do conjunto. Usa as colunas x, y."""
    cx, cy = df["x"].median(), df["y"].median()
    return float(np.median(np.hypot(df["x"] - cx, df["y"] - cy)))


def escolher_lugar(df, bairro_key):
    """Entre os pontos/trechos de ruas de mesmo nome, devolve (subconjunto, bairro_ok).

    1. se há bairro e algum loteamento compatível: só esses (bairro_ok = True);
    2. senão, se tudo está concentrado (dispersão <= DISPERSAO_MAX_M): todos (bairro_ok = False);
    3. senão: (None, False) -> nomes iguais em lugares diferentes, não dá para escolher.
    """
    if df is None or df.empty:
        return None, False
    if bairro_key:
        ok = df["lote_key"].map(lambda l: bairro_compativel(bairro_key, l))
        if ok.any():
            return df[ok], True
    if dispersao(df) <= DISPERSAO_MAX_M:
        return df, False
    return None, False


def dentro_da_area(lat, lon):
    """True se a coordenada está dentro da área do município (envelope do cnefe + folga)."""
    return BBOX[0] <= lon <= BBOX[2] and BBOX[1] <= lat <= BBOX[3]


# --- Teste rápido da leitura do número de referência ----------------------------------------------------
for _txt, _g in [("2170", True), ("de 6001/6002 ao fim", False), ("até 999/1000", False),
                 ("de 1001/1002 a 1999/2000", False), ("lado ímpar", False), ("km 141,001 ao km 150,000", False)]:
    log.info("  numero_referencia(%r, grande=%s) = %s", _txt, _g, numero_referencia(_txt, _g))
log.info("Funções auxiliares definidas!")

## 3. CEPs alvo e CSV existente
**Alvo** = CEPs da RAIS válidos do município (8 dígitos, prefixo `12`; os terminados em `000` **também** entram, pois muitos são CEPs reais de avenidas) que:
- são de **grande usuário** (sufixo ≥ 900), com ou sem vínculos; **ou**
- **não** estão na coluna `cep` da base **e** **não** têm pontos no CNEFE.

**CSV existente:** linhas `manual` são sempre mantidas; linhas com coordenada são mantidas se o CEP ainda é alvo. O resto é refeito.

In [ ]:
log.info("Selecionando os CEPs alvo...")
rais = pd.read_parquet(ARQ_RAIS, columns=["cep", "qtd_vinc_ativos"])
rais["cep"] = rais["cep"].astype(object).where(rais["cep"].notna(), "").astype(str).str.strip()
rais["qtd_vinc_ativos"] = rais["qtd_vinc_ativos"].fillna(0)
por_cep = (rais.groupby("cep", as_index=False)
           .agg(n_estab_soma_anos=("cep", "size"), vinc_ativos_soma_anos=("qtd_vinc_ativos", "sum")))
por_cep["vinc_ativos_soma_anos"] = por_cep["vinc_ativos_soma_anos"].astype("int64")

base = pyogrio.read_dataframe(ARQ_BASE, layer=CAMADA_BASE, columns=["id", "nome", "loteamento", "cep"])
for col in ["nome", "loteamento", "cep"]:
    base[col] = base[col].astype(object).where(base[col].notna(), "").astype(str).str.strip()
ceps_base = set(base.loc[base["cep"] != "", "cep"].str.split("|").explode().str.replace(r"\D", "", regex=True))
locais = pd.read_parquet(ARQ_LOCAIS, columns=["cep", "lat", "lon", "id_seg"])
ceps_cnefe = set(locais["cep"])

sufixo = por_cep["cep"].str[-3:]
valido = por_cep["cep"].str.fullmatch(r"\d{8}") & por_cep["cep"].str.startswith(PREFIXO_MUNICIPIO)
grande = valido & (pd.to_numeric(sufixo, errors="coerce") >= SUFIXO_ESPECIAL_MIN)
sem_lugar = valido & ~por_cep["cep"].isin(ceps_base) & ~por_cep["cep"].isin(ceps_cnefe)
alvos = por_cep[grande | sem_lugar].copy()
alvos["grande_usuario"] = grande[grande | sem_lugar].to_numpy()

# --- Universo dos correios (logradouro e bairro por CEP) --------------------------------------------------------
ponte = pd.read_parquet(ARQ_PONTE, columns=["cep", "logradouro_correios", "bairro_correios"])
orfaos = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
correios = pd.concat([
    ponte.groupby("cep", as_index=False).agg(logradouro=("logradouro_correios", "first"), bairro=("bairro_correios", "first")),
    orfaos.rename(columns={"logradouro_orig": "logradouro", "bairro_orig": "bairro"})[["cep", "logradouro", "bairro"]],
]).drop_duplicates("cep")
correios["logradouro"] = correios["logradouro"].astype(str).str.split(" | ", regex=False).str[0]   # grafias alternativas: fica a primeira
alvos["grupo"] = np.select([alvos["grande_usuario"], alvos["cep"].isin(set(correios["cep"]))],
                           ["grande_usuario", "correios_sem_trecho"], default="fora_dos_correios")
alvos = alvos.sort_values("vinc_ativos_soma_anos", ascending=False).reset_index(drop=True)
log.info("CEPs alvo: %d | por grupo:\n%s", len(alvos), alvos["grupo"].value_counts().to_string())

# --- CSV existente (ou o antigo de grandes usuários, só as linhas 'manual') ----------------------------------------
if ARQ_CSV.exists():
    existente = fontes.ler_csv_editado(ARQ_CSV)                       # aceita o que o Excel grava (UTF-8, ANSI, MS-DOS)
elif ARQ_CSV_ANTIGO.exists():
    antigo = fontes.ler_csv_editado(ARQ_CSV_ANTIGO)
    existente = antigo[antigo.get("fonte_coord", pd.Series("", index=antigo.index)) == "manual"].copy()
    existente = existente.rename(columns={"numero": "numero_ref"})
    log.info("CSV antigo de grandes usuários: %d linhas 'manual' trazidas.", len(existente))
    if "cep_novo" in antigo.columns:                                   # anotações de CEP novo também valem
        CEPS_SUBSTITUIDOS.update({c: n for c, n in zip(antigo["cep"], antigo["cep_novo"].str.replace(r"\D", "", regex=True)) if len(n) == 8})
else:
    existente = pd.DataFrame(columns=COLUNAS_CSV)
for col in COLUNAS_CSV:
    if col not in existente.columns:
        existente[col] = ""
existente = existente[COLUNAS_CSV].drop_duplicates("cep")
for col in ["lat", "lon"]:                                          # conserta decimal estragado pelo Excel (-23.220.157 -> -23.220157)
    existente[col] = fontes.ler_coordenada(existente[col]).map(lambda v: "" if pd.isna(v) else f"{v:.6f}")

# --- CEP novo: coluna cep_novo do CSV (prevalece) + dicionário da configuração --------------------------------------
tokens_csv = {c: ler_ceps_novos(t) for c, t in zip(existente["cep"], existente["cep_novo"])}       # CEPs de 8 dígitos de cada célula
novo_csv = {c: t[0] for c, t in tokens_csv.items() if len(t) == 1}                                  # 1 CEP = troca de CEP
DESMEMBRADOS = {c: t for c, t in tokens_csv.items() if len(t) >= 2}                                 # 2 ou mais = CEP desmembrado
mapa_novo = {**CEPS_SUBSTITUIDOS, **novo_csv}
# --- Endereços escritos à mão no CSV (fonte_endereco = manual): usados no lugar das APIs ------------------------------------
ENDERECO_MANUAL = {r.cep: r for r in existente[existente["fonte_endereco"].str.lower() == "manual"].itertuples(index=False)
                   if r.logradouro.strip()}
log.info("Do CSV: %d CEPs com endereço manual | %d CEPs desmembrados | %d CEPs trocados", len(ENDERECO_MANUAL), len(DESMEMBRADOS), len(novo_csv))
manual = existente["fonte_coord"].str.lower() == "manual"
# linha com coordenada, de CEP sem endereço dos correios e sem cidade registrada, é refeita: confere a cidade (as respostas estão no cache)
sem_cidade = (existente["cidade"].str.strip() == "") & ~existente["fonte_endereco"].isin(["correios", "nao_necessario"])
com_coord = (existente["lat"].str.strip() != "") & existente["cep"].isin(set(alvos["cep"])) & ~sem_cidade
sem_ponto_sabido = existente["fonte_coord"].isin(["outra_cidade", "nao_localizavel"]) & existente["cep"].isin(set(alvos["cep"]))   # já sabido: não refaz
mantidas = existente[manual | com_coord | sem_ponto_sabido].copy()
pendentes = alvos[~alvos["cep"].isin(set(mantidas["cep"]))].copy()
pendentes["cep_busca"] = pendentes["cep"].map(lambda c: mapa_novo.get(c, c))      # CEP usado nas consultas
log.info("CSV: %d linhas | mantidas (manual ou com coordenada): %d | CEPs a localizar agora: %d", len(existente), len(mantidas), len(pendentes))
validar_tabela("CEPs alvo", alvos, chave=["cep"], obrigatorias=["cep", "grupo"])
log.info("CEPs alvo prontos!")

## 4. Endereço de cada CEP pendente
- **Correios** (grupo `correios_sem_trecho`): rua e bairro da lista da etapa 02, sem consulta;
- grande usuário que **já tem pontos no CNEFE** com o CEP: sem consulta (a etapa 7 usa os pontos);
- senão **ViaCEP** (nome do estabelecimento, rua, complemento e bairro);
- se o ViaCEP não achar, **BrasilAPI** (só rua e bairro; a coordenada dela não é usada).

Respostas já consultadas vêm do cache. Se um serviço bloquear, o CEP fica `nao_consultado` e é refeito na próxima execução.

In [ ]:
log.info("Buscando o endereço de %d CEPs...", len(pendentes))
info_correios = correios.set_index("cep")
enderecos = []
for i, linha in enumerate(pendentes.itertuples(index=False), start=1):            # laço curto (só os pendentes)
    reg = {"cep": linha.cep, "cep_busca": linha.cep_busca, "unidade": "", "logradouro": "", "complemento": "",
           "bairro": "", "cidade": "", "fonte_endereco": "nao_encontrado"}
    if linha.cep in DESMEMBRADOS:                                              # 0) CEP desmembrado: usa os CEPs novos (sem consulta)
        reg.update(fonte_endereco="desmembrado", cidade=MUNICIPIO)
    elif linha.cep in ENDERECO_MANUAL:                                         # 0) endereço escrito à mão no CSV (sem consulta)
        m = ENDERECO_MANUAL[linha.cep]
        reg.update(unidade=m.unidade, logradouro=m.logradouro, complemento=m.complemento, bairro=m.bairro,
                   cidade=m.cidade.strip() or MUNICIPIO, fonte_endereco="manual")
    elif linha.cep_busca in info_correios.index:                               # 1) correios (sem consulta)
        rua, faixa = separar_faixa(info_correios.at[linha.cep_busca, "logradouro"])
        reg.update(logradouro=rua, complemento=faixa, bairro=info_correios.at[linha.cep_busca, "bairro"], cidade=MUNICIPIO, fonte_endereco="correios")
    elif linha.grande_usuario and linha.cep_busca in ceps_cnefe:               # 2) grande usuário com pontos no cnefe: sem API
        reg.update(fonte_endereco="nao_necessario", cidade=MUNICIPIO)           # tem pontos no cnefe do município
    else:
        dados, status_v = VIACEP.consultar(linha.cep_busca, URL_VIACEP.format(cep=linha.cep_busca))   # 3) viacep
        if status_v == "ok" and isinstance(dados, dict) and not dados.get("erro"):
            reg.update(unidade=dados.get("unidade", ""), logradouro=dados.get("logradouro", ""),
                       complemento=dados.get("complemento", ""), bairro=dados.get("bairro", ""),
                       cidade=dados.get("localidade", ""), fonte_endereco="viacep")
        else:                                                                  # 4) brasilapi, só se o viacep não deu endereço
            dados, status_b = BRASILAPI.consultar(linha.cep_busca, URL_BRASILAPI.format(cep=linha.cep_busca))
            if status_b == "ok" and isinstance(dados, dict) and dados.get("street"):
                reg.update(logradouro=dados.get("street", ""), bairro=dados.get("neighborhood", ""),
                           cidade=dados.get("city", ""), fonte_endereco="brasilapi")
            elif status_v in FALHAS or status_b in FALHAS:                     # não deu para consultar: refazer depois
                reg["fonte_endereco"] = "nao_consultado"
    enderecos.append(reg)
    if i % 25 == 0 or i == len(pendentes):
        log.info("  %d/%d endereços | consultas na rede: viacep %d, brasilapi %d | vindas do cache: %d",
                 i, len(pendentes), VIACEP.n_rede, BRASILAPI.n_rede, VIACEP.n_cache + BRASILAPI.n_cache)
end = pd.DataFrame(enderecos, columns=["cep", "cep_busca", "unidade", "logradouro", "complemento", "bairro", "cidade", "fonte_endereco"])
end = end.merge(pendentes[["cep", "grande_usuario", "grupo"]], on="cep", how="left")
ref = [numero_referencia(c, g) for c, g in zip(end["complemento"], end["grande_usuario"])]
end["numero_ref"] = [r[0] for r in ref]
end["tipo_numero"] = [r[1] for r in ref]
end["nome_key"] = normalizar(end["logradouro"]).map(tirar_tipo)
end["bairro_key"] = normalizar(end["bairro"])
log.info("Fonte do endereço:\n%s", end["fonte_endereco"].value_counts().to_string() if len(end) else "(nada a buscar)")
n_outra = int(end["cidade"].map(de_outro_municipio).sum()) if len(end) else 0
log.info("CEPs de OUTRO município (não serão localizados): %d%s", n_outra,
         "\n" + end.loc[end["cidade"].map(de_outro_municipio), "cidade"].value_counts().head(10).to_string() if n_outra else "")
n_nao_consultado = int((end["fonte_endereco"] == "nao_consultado").sum()) if len(end) else 0
if n_nao_consultado:
    log.warning("ATENÇÃO: %d CEPs ficaram SEM endereço porque a consulta às APIs não foi possível. Rode a etapa de novo mais tarde.", n_nao_consultado)
log.info("Endereços prontos!")

## 5. CNEFE e Geosanja preparados para a busca por endereço
- **CNEFE:** cada endereço com o nome da rua comparável, o número e o **loteamento** do trecho a que o local foi ligado na etapa 03 (para escolher entre ruas de mesmo nome);
- **Geosanja:** cada trecho da base com nome comparável, loteamento e centro (em metros).

In [ ]:
log.info("Carregando o cnefe com números...")
def ler_cnefe(codificacao):
    """Lê o cnefe (separador ';') só com as colunas necessárias, tudo como texto."""
    return pd.read_csv(ARQ_CNEFE, sep=";", dtype=str, usecols=COLUNAS_CNEFE, encoding=codificacao, keep_default_na=False)

try:
    cnefe = ler_cnefe("utf-8")
except UnicodeDecodeError:
    log.warning("Arquivo não está em UTF-8; relendo como Latin-1.")
    cnefe = ler_cnefe("latin-1")
cnefe["cep"] = cnefe["CEP"].str.replace(r"\D", "", regex=True)
cnefe["lat"] = pd.to_numeric(cnefe["LATITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["lon"] = pd.to_numeric(cnefe["LONGITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe = cnefe[cnefe["lat"].notna() & cnefe["lon"].notna()].copy()
cnefe["num"] = pd.to_numeric(cnefe["NUM_ENDERECO"], errors="coerce").where(lambda s: s > 0)     # 0 = sem número
cnefe["nome_key"] = normalizar(cnefe["NOM_TITULO_SEGLOGR"] + " " + cnefe["NOM_SEGLOGR"]).map(tirar_tipo)
BBOX = (cnefe["lon"].min() - MARGEM_GRAUS, cnefe["lat"].min() - MARGEM_GRAUS, cnefe["lon"].max() + MARGEM_GRAUS, cnefe["lat"].max() + MARGEM_GRAUS)

# --- Loteamento do trecho a que cada local foi ligado (etapa 03) ----------------------------------------------------
lote_seg = dict(zip(base["id"], normalizar(base["loteamento"])))
locais["lote_key"] = locais["id_seg"].map(lambda i: lote_seg.get(i, "") if pd.notna(i) else "")
cnefe = cnefe.merge(locais[["cep", "lat", "lon", "lote_key"]].drop_duplicates(["cep", "lat", "lon"]), on=["cep", "lat", "lon"], how="left")
cnefe["lote_key"] = cnefe["lote_key"].fillna("")
xy = gpd.GeoSeries(gpd.points_from_xy(cnefe["lon"], cnefe["lat"]), crs=CRS_GRAUS).to_crs(CRS_METRICO)
cnefe["x"], cnefe["y"] = xy.x.to_numpy(), xy.y.to_numpy()
cnefe = cnefe[["cep", "nome_key", "num", "lat", "lon", "x", "y", "lote_key"]]
log.info("cnefe: %d endereços | com número: %d | com loteamento: %d | área (lon/lat): %s", len(cnefe),
         int(cnefe["num"].notna().sum()), int((cnefe["lote_key"] != "").sum()), tuple(round(float(v), 3) for v in BBOX))

log.info("Preparando os trechos da base...")
segs = base[["id", "nome", "loteamento", "geometry"]].copy()
segs["nome_key"] = normalizar(segs["nome"]).map(tirar_tipo)
segs = segs[(segs["nome_key"] != "") & ~segs["nome_key"].str.match(r"^SEM NOME\b")].copy()
segs["lote_key"] = normalizar(segs["loteamento"])
segs["x"], segs["y"] = segs.geometry.centroid.x.to_numpy(), segs.geometry.centroid.y.to_numpy()
log.info("Trechos com nome: %d", len(segs))

cnefe_por_cep = {k: v for k, v in cnefe[cnefe["cep"].isin(set(end["cep_busca"]))].groupby("cep")}   # endereços por CEP
# --- Apoio aos dois últimos recursos: endereços por bairro (loteamento) e por prefixo de CEP -------------------------------
lotes_cnefe = [l for l in cnefe["lote_key"].unique() if l]                           # loteamentos que aparecem nos endereços
cnefe["prefixo6"] = cnefe["cep"].str[:6]
cnefe_por_prefixo = {k: v for k, v in cnefe[cnefe["prefixo6"].isin(set(end["cep_busca"].str[:6]))].groupby("prefixo6")}
cnefe_desmembrado = cnefe[cnefe["cep"].isin({c for lista in DESMEMBRADOS.values() for c in lista})]   # endereços dos CEPs novos
log.info("Loteamentos no cnefe: %d | prefixos de CEP a consultar: %d", len(lotes_cnefe), len(cnefe_por_prefixo))
log.info("CNEFE e Geosanja prontos!")

## 6. Nomes de rua compatíveis
Compara o nome de cada endereço buscado com **todos** os nomes de rua do CNEFE e do Geosanja de uma vez (matriz de similaridade). Compatível = `token_sort_ratio` ≥ `cnefe.limiar_nome` **ou** um nome contido no outro (o menor com ≥ 2 palavras), a mesma regra da etapa 03.

In [ ]:
def mapa_compativeis(alvos_nomes, candidatos):
    """Para cada nome buscado, a lista de nomes candidatos compatíveis."""
    alvos_nomes = [a for a in dict.fromkeys(alvos_nomes) if a]
    candidatos = [c for c in dict.fromkeys(candidatos) if c]
    if not alvos_nomes or not candidatos:
        return {}
    ordem = process.cdist(alvos_nomes, candidatos, scorer=fuzz.token_sort_ratio, workers=-1)   # ordem das palavras não importa
    contido = process.cdist(alvos_nomes, candidatos, scorer=fuzz.token_set_ratio, workers=-1)  # um nome dentro do outro
    menor = np.minimum.outer(np.array([len(a.split()) for a in alvos_nomes]), np.array([len(c.split()) for c in candidatos]))
    ok = (ordem >= LIMIAR_NOME) | ((contido == 100) & (menor >= 2))
    return {a: [candidatos[j] for j in np.flatnonzero(ok[i])] for i, a in enumerate(alvos_nomes)}


log.info("Comparando nomes de rua...")
compat_cnefe = mapa_compativeis(end["nome_key"], cnefe["nome_key"].unique())
compat_geo = mapa_compativeis(end["nome_key"], segs["nome_key"].unique())
cnefe_por_nome = {k: v for k, v in cnefe[cnefe["nome_key"].isin({c for l in compat_cnefe.values() for c in l})].groupby("nome_key")}
geo_por_nome = {k: v for k, v in segs[segs["nome_key"].isin({c for l in compat_geo.values() for c in l})].groupby("nome_key")}
log.info("Nomes buscados: %d | com rua compatível no cnefe: %d | no geosanja: %d",
         end["nome_key"].replace("", np.nan).nunique(), sum(1 for v in compat_cnefe.values() if v), sum(1 for v in compat_geo.values() if v))
log.info("Nomes comparados!")

## 7. Localizar
Para cada CEP, tenta as fontes na ordem da tabela do topo e para na primeira que achar (`cnefe_bairro` e `cnefe_cep_vizinhos` por último). O Nominatim só é consultado se CNEFE e Geosanja falharem (no máximo 1 consulta por segundo; respostas já feitas vêm do cache), e o resultado só vale se a rua devolvida tiver nome compatível.

In [ ]:
def juntar(dic, chaves):
    """Junta as tabelas de várias ruas compatíveis numa só (None se não houver nenhuma)."""
    partes = [dic[k] for k in chaves if k in dic]
    return pd.concat(partes) if partes else None


def ponto_cnefe(df):
    """Endereço do conjunto mais próximo do centro (mediana): fica em cima da rua, não no meio da quadra."""
    cx, cy = df["x"].median(), df["y"].median()
    melhor = df.iloc[int(np.argmin(np.hypot(df["x"] - cx, df["y"] - cy)))]
    return float(melhor["lat"]), float(melhor["lon"])


@lru_cache(maxsize=None)
def cnefe_do_bairro(bairro_key):
    """Endereços do cnefe cujo loteamento é compatível com o bairro. None se faltar compatível, houver < 3 ou estiverem espalhados."""
    compativeis = [l for l in lotes_cnefe if bairro_compativel(bairro_key, l)]
    if not compativeis:
        return None
    pts = cnefe[cnefe["lote_key"].isin(compativeis)]
    return pts if len(pts) >= 3 and dispersao(pts) <= DISPERSAO_MAX_M else None


def nominatim(logradouro, numero, bairro, nome_key):
    """Busca no Nominatim (do mais completo ao mais simples). Devolve ((lat, lon, fonte) ou None, consulta_completa).

    consulta_completa = False se alguma busca não pôde ser feita (serviço bloqueado ou sem rede)."""
    completa = True
    viewbox = f"{BBOX[0]},{BBOX[3]},{BBOX[2]},{BBOX[1]}"                      # área do município (oeste, norte, leste, sul)
    consultas = []
    if numero and bairro:
        consultas.append(f"{logradouro}, {numero}, {bairro}, {CIDADE}")
    if bairro:
        consultas.append(f"{logradouro}, {bairro}, {CIDADE}")
    consultas.append(f"{logradouro}, {CIDADE}")
    for q in consultas:
        dados, status = NOMINATIM.consultar(q, URL_NOMINATIM, params={"q": q, "format": "jsonv2", "addressdetails": 1, "limit": 5,
                                                                      "countrycodes": "br", "viewbox": viewbox, "bounded": 1})
        if status in FALHAS:
            completa = False
        for r in (dados or []) if status == "ok" else []:
            endereco = r.get("address", {})
            chave = nome_chave_txt(endereco.get("road", ""))                    # nome da rua devolvida pelo Nominatim
            if chave and (chave == nome_key or mapa_compativeis([chave], [nome_key]).get(chave)):   # mesma rua?
                lat, lon = float(r["lat"]), float(r["lon"])
                if dentro_da_area(lat, lon):
                    return (lat, lon, "geocoding_numero" if endereco.get("house_number") else "geocoding_rua"), completa
    return None, completa


def localizar(e):
    """Devolve (lat, lon, fonte_coord, confianca, n_pontos, obs) para um endereço (linha de 'end')."""
    if e.fonte_endereco == "desmembrado":                                          # CEP desmembrado: centro dos endereços dos CEPs novos
        novos_ceps = DESMEMBRADOS[e.cep]
        pts = cnefe_desmembrado[cnefe_desmembrado["cep"].isin(novos_ceps)]
        if pts.empty:
            return np.nan, np.nan, "sem_coordenada", "", 0, "CEP desmembrado: nenhum dos CEPs novos tem endereços no CNEFE"
        lat, lon = ponto_cnefe(pts)
        disp = dispersao(pts)
        return lat, lon, "cnefe_ceps_desmembrados", "media" if disp <= DISPERSAO_MAX_M else "baixa", len(pts), \
            f"CEP desmembrado em {len(novos_ceps)} CEPs ({pts['cep'].nunique()} com endereços no CNEFE); dispersão {disp:.0f} m"
    if de_outro_municipio(e.cidade):                                               # CEP de outra cidade: não localiza
        return np.nan, np.nan, "outra_cidade", "", 0, f"CEP de {e.cidade} (fora do município): não localizado"
    num, tipo_num = e.numero_ref, e.tipo_numero
    tem_num = num is not None and not (isinstance(num, float) and np.isnan(num))
    por_cep = cnefe_por_cep.get(e.cep_busca)
    rua_cnefe, bairro_ok = escolher_lugar(juntar(cnefe_por_nome, compat_cnefe.get(e.nome_key, [])), e.bairro_key)
    conf_rua = "media" if bairro_ok else "baixa"
    # 1) mesmo CEP e mesmo número (número de prédio)
    if por_cep is not None and tem_num and tipo_num == "predio":
        m = por_cep[por_cep["num"] == num]
        if len(m):
            return m["lat"].mean(), m["lon"].mean(), "cnefe_cep_numero", "alta", len(m), ""
    # 2) mesma rua e mesmo número (número de prédio)
    if rua_cnefe is not None and tem_num and tipo_num == "predio":
        m = rua_cnefe[rua_cnefe["num"] == num]
        if len(m):
            return m["lat"].mean(), m["lon"].mean(), "cnefe_endereco", "alta" if bairro_ok else "media", len(m), ""
    # 3) endereços do cnefe com esse CEP
    if por_cep is not None:
        return por_cep["lat"].median(), por_cep["lon"].median(), "cnefe_cep", "media", len(por_cep), ""
    # 4) mesma rua, número mais próximo (prédio ou meio da faixa)
    if rua_cnefe is not None and tem_num and rua_cnefe["num"].notna().any():
        dif = (rua_cnefe["num"] - num).abs()
        melhor = rua_cnefe[dif == dif.min()]
        return melhor["lat"].mean(), melhor["lon"].mean(), "cnefe_rua_numero_proximo", conf_rua, len(melhor), \
            f"número de referência {int(num)} ({tipo_num}); usado o {int(melhor['num'].iloc[0])}"
    # 5) centro dos endereços da rua
    if rua_cnefe is not None:
        lat, lon = ponto_cnefe(rua_cnefe)
        return lat, lon, "cnefe_rua", conf_rua, len(rua_cnefe), ""
    # 6) trechos da base com nome compatível
    trechos, bairro_ok_g = escolher_lugar(juntar(geo_por_nome, compat_geo.get(e.nome_key, [])), e.bairro_key)
    if trechos is not None:
        uniao = shapely.union_all(trechos.geometry.to_numpy())
        pt = shapely.ops.nearest_points(uniao, uniao.centroid)[0]                 # ponto da linha mais perto do centro
        lon, lat = para_graus.transform(pt.x, pt.y)
        return lat, lon, "geosanja_nome", "media" if bairro_ok_g else "baixa", len(trechos), ""
    # 7) Nominatim
    incompleta = e.fonte_endereco == "nao_consultado"                              # endereço nem chegou a ser consultado
    if e.logradouro:
        achado, completa = nominatim(e.logradouro, int(num) if tem_num else None, e.bairro, e.nome_key)
        if achado:
            lat, lon, fonte = achado
            return lat, lon, fonte, "media" if fonte == "geocoding_numero" else "baixa", 0, ""
        incompleta = incompleta or not completa
    if incompleta:
        return np.nan, np.nan, "sem_coordenada", "", 0, "consulta às APIs NÃO concluída (serviço indisponível): rode a etapa 04 de novo"
    # 8) último recurso: centro do bairro (endereços do cnefe no mesmo loteamento)
    pts_bairro = cnefe_do_bairro(e.bairro_key) if e.bairro_key else None
    if pts_bairro is not None:
        lat, lon = ponto_cnefe(pts_bairro)
        return lat, lon, "cnefe_bairro", "baixa", len(pts_bairro), f"aproximado pelo centro do bairro {e.bairro}"
    # 9) último recurso: CEPs vizinhos (mesmos 6 primeiros dígitos). Não vale para CEP terminado em 000 (pode ser CEP geral)
    viz = cnefe_por_prefixo.get(e.cep_busca[:6]) if not e.cep_busca.endswith("000") else None
    if viz is not None and len(viz) >= 3 and dispersao(viz) <= DISPERSAO_MAX_M:
        lat, lon = ponto_cnefe(viz)
        return lat, lon, "cnefe_cep_vizinhos", "baixa", len(viz), f"aproximado pelos CEPs vizinhos {e.cep_busca[:6]}xx ({viz['cep'].nunique()} CEPs)"
    motivo = "endereço não encontrado nas APIs" if e.fonte_endereco == "nao_encontrado" else "endereço sem correspondência"
    return np.nan, np.nan, "sem_coordenada", "", 0, f"{motivo}: preencher lat/lon e fonte_coord = manual"


log.info("Localizando %d CEPs...", len(end))
resultados = []
for i, e in enumerate(end.itertuples(index=False), start=1):                       # laço curto (só os pendentes)
    lat, lon, fonte, conf, n_pts, obs = localizar(e)
    if e.cep_busca != e.cep:
        obs = f"CEP substituído por {e.cep_busca}" + (f"; {obs}" if obs else "")
    resultados.append({"cep": e.cep, "lat": lat, "lon": lon, "fonte_coord": fonte, "confianca": conf, "n_pontos": n_pts, "obs": obs})
    if i % 25 == 0 or i == len(end):
        log.info("  %d/%d localizados", i, len(end))
novos = end.merge(pd.DataFrame(resultados, columns=["cep", "lat", "lon", "fonte_coord", "confianca", "n_pontos", "obs"]), on="cep", how="left")
novos["cep_novo"] = novos["cep"].map(lambda c: "|".join(DESMEMBRADOS[c]) if c in DESMEMBRADOS else mapa_novo.get(c, ""))
novos["numero_ref"] = novos["numero_ref"].map(lambda v: "" if v is None or (isinstance(v, float) and np.isnan(v)) else str(int(v)))
log.info("Fontes das coordenadas (novos):\n%s", novos["fonte_coord"].value_counts().to_string() if len(novos) else "(nada novo)")
log.info("Localização concluída!")

## 8. Exportação
Junta as linhas mantidas com as novas, atualiza estabelecimentos e vínculos e grava o CSV (ordenado por vínculos). Se o CSV já existia, guarda antes uma **cópia de segurança** com data e hora.

In [ ]:
log.info("Gravando o CSV de CEPs localizados...")
for col in ["lat", "lon"]:
    novos[col] = novos[col].map(lambda v: "" if pd.isna(v) else f"{v:.6f}")       # texto com ponto decimal
novos["n_pontos"] = novos["n_pontos"].astype(str)
final = pd.concat([mantidas[COLUNAS_CSV].astype(str), novos.reindex(columns=COLUNAS_CSV).fillna("").astype(str)], ignore_index=True)
final.loc[final["fonte_coord"].str.lower() == "manual", "confianca"] = "manual"
final = final.drop(columns=["grupo", "n_estab_soma_anos", "vinc_ativos_soma_anos"]).merge(
    alvos[["cep", "grupo", "n_estab_soma_anos", "vinc_ativos_soma_anos"]], on="cep", how="left")
final["grupo"] = final["grupo"].fillna("nao_e_mais_alvo")                          # ex.: manual de um CEP que já ganhou trecho
for col in ["n_estab_soma_anos", "vinc_ativos_soma_anos"]:
    final[col] = final[col].fillna(0).astype(int)
final = final.sort_values("vinc_ativos_soma_anos", ascending=False)[COLUNAS_CSV].reset_index(drop=True)

if ARQ_CSV.exists() and len(novos) == 0:
    log.info("Nada novo: o CSV existente foi mantido sem alterações.")
else:
    if ARQ_CSV.exists():
        fontes.backup(cfg, ARQ_CSV)                                # cópia datada em 02_dados_editados/_backup
    ARQ_CSV.parent.mkdir(parents=True, exist_ok=True)
    final.to_csv(ARQ_CSV, sep=";", index=False, encoding="utf-8-sig")
    log.info("CSV salvo: %s (%d linhas)!", ARQ_CSV.name, len(final))
validar_tabela("CSV de CEPs localizados", final, chave=["cep"], obrigatorias=["cep", "grupo"])
log.info("Exportação concluída!")

## 9. Resumo

In [ ]:
log.info("Resumo final...")
com_coord = final["lat"].str.strip() != ""
log.info("CEPs no CSV: %d | com coordenada: %d | sem coordenada: %d", len(final), int(com_coord.sum()), int((~com_coord).sum()))
log.info("Por fonte da coordenada:\n%s", final["fonte_coord"].value_counts().to_string())
log.info("Por confiança:\n%s", final.loc[com_coord, "confianca"].value_counts().to_string())
log.info("Por grupo (com / sem coordenada):\n%s", pd.crosstab(final["grupo"], com_coord.map({True: "com", False: "sem"})).to_string())
sem_ponto_sabido_final = final["fonte_coord"].isin(["outra_cidade", "nao_localizavel"])
log.info("CEPs sem ponto e sem pendência manual: %d (outro município: %d | marcados como não localizáveis: %d)",
         int(sem_ponto_sabido_final.sum()), int((final["fonte_coord"] == "outra_cidade").sum()), int((final["fonte_coord"] == "nao_localizavel").sum()))
sem = final[~com_coord & ~sem_ponto_sabido_final].head(15)
if len(sem):
    log.info("Maiores sem coordenada (preencher lat/lon e fonte_coord = manual):\n%s",
             sem[["cep", "grupo", "unidade", "logradouro", "bairro", "fonte_endereco", "vinc_ativos_soma_anos"]].to_string(index=False))
for _s in SERVICOS:
    log.info("%s: %d consultas na rede | %d respostas do cache | %d chaves não consultadas | desligado: %s | cache: %d respostas",
             _s.nome, _s.n_rede, _s.n_cache, len(_s.pendentes), _s.desligado, len(_s.cache))
n_incompletos = int(final["obs"].str.contains("NÃO concluída", regex=False).sum())
if n_incompletos or any(_s.pendentes for _s in SERVICOS):
    log.warning("ATENÇÃO: NÃO FOI POSSÍVEL CONSULTAR TUDO. %d CEPs ficaram sem coordenada por falha/bloqueio das APIs "
                "(ver coluna obs do CSV). Rode a etapa 04 de novo mais tarde: o que já foi consultado está no cache e não será repetido.", n_incompletos)
log.info("Próximo: etapas 05 (classe G) e 06.")
log.info("Etapa 04 concluída!")